# fmcib embedding cho CT lesion (chạy trên Google Colab)

**Mục đích:** trích embedding từ **fmcib** (Foundation Model for Cancer Imaging Biomarkers) — model 3D chuyên biệt cho tổn thương CT — mà máy Windows local không cài được.

**Bạn cần làm:**
1. `Runtime -> Change runtime type -> GPU` (T4 miễn phí).
2. Upload 2 file `fmcib_patches_discovery.npz` và `fmcib_patches_rad_valid.npz` (tôi cắt sẵn tại local — mỗi patch là 50x50x50 @1mm quanh tâm lesion, chuẩn fmcib) lên Google Drive, ví dụ thư mục `MyDrive/fmcib/`.
3. Chạy các cell từ trên xuống.
4. Sau khi xong, tải 2 file `fmcib_embed_*.parquet` trong `MyDrive/fmcib/` về, gửi lại cho tôi để chạy eval (discovery-CV + external + fusion) trên pipeline đã kiểm chứng.

Không cần upload CT gốc (hàng chục GB) — chỉ 2 file patch nhỏ.

In [ ]:
# 1) Cài fmcib (Colab = Linux nên build được, khác Windows)
!pip install -q foundation-cancer-image-biomarker
import torch; print('CUDA:', torch.cuda.is_available(), torch.cuda.get_device_name(0) if torch.cuda.is_available() else '')

In [ ]:
# 2) Mount Google Drive + đặt đường dẫn
from google.colab import drive
drive.mount('/content/drive')
DIR = '/content/drive/MyDrive/fmcib'   # <-- sửa nếu bạn để chỗ khác
import os; print('files:', os.listdir(DIR))

In [ ]:
# 3) Hàm trích embedding: patch npz -> viết .nii tạm (1mm) -> fmcib get_features
import numpy as np, pandas as pd, SimpleITK as sitk, os, tempfile
from fmcib.run import get_features

def embed_npz(npz_path, out_parquet):
    d = np.load(npz_path, allow_pickle=True)
    patches, pids = d['patches'], d['patients']
    tmp = tempfile.mkdtemp(); rows = []
    for p, pid in zip(patches, pids):
        img = sitk.GetImageFromArray(p.astype('int16'))   # [z,y,x]
        img.SetSpacing((1.0, 1.0, 1.0))                   # origin (0,0,0) mac dinh
        fn = os.path.join(tmp, f'{pid}.nii.gz'); sitk.WriteImage(img, fn)
        # patch 50^3 @1mm, tam lesion ~ voxel 25 -> world (25,25,25)
        rows.append({'image_path': fn, 'coordX': 25.0, 'coordY': 25.0, 'coordZ': 25.0, 'PatientID': str(pid)})
    csv = os.path.join(tmp, 'seeds.csv'); pd.DataFrame(rows).to_csv(csv, index=False)
    feat = get_features(csv)
    # lay cac cot feature (bo cot input)
    drop = {'image_path', 'coordX', 'coordY', 'coordZ', 'PatientID'}
    fcols = [c for c in feat.columns if c not in drop and pd.api.types.is_numeric_dtype(feat[c])]
    emb = feat[fcols].values
    ids = feat['PatientID'].astype(str).values if 'PatientID' in feat else [r['PatientID'] for r in rows]
    out = pd.DataFrame(emb, index=ids, columns=[f'fmcib_{j}' for j in range(emb.shape[1])])
    out.index.name = 'patient'; out.to_parquet(out_parquet)
    print(f'{os.path.basename(out_parquet)}: {out.shape}')
    return out.shape

In [ ]:
# 4) Chay cho ca 2 cohort -> luu parquet vao Drive
embed_npz(f'{DIR}/fmcib_patches_discovery.npz', f'{DIR}/fmcib_embed_discovery.parquet')
embed_npz(f'{DIR}/fmcib_patches_rad_valid.npz', f'{DIR}/fmcib_embed_rad_valid.parquet')
print('XONG. Tai 2 file fmcib_embed_*.parquet trong Drive ve, gui lai cho Claude.')

## Nếu cell 3-4 lỗi (API fmcib khác phiên bản)

Thử cách gọi model trực tiếp (fallback):
```python
from fmcib.models import fmcib_model
import torch, numpy as np, pandas as pd
model = fmcib_model().eval().cuda()
def embed_direct(npz, out):
    d = np.load(npz, allow_pickle=True); P, ids = d['patches'], d['patients']
    vecs = []
    for p in P:
        x = np.clip(p.astype('float32'), -1024, 3071)
        x = (x - x.mean())/(x.std()+1e-6)
        t = torch.tensor(x)[None,None].cuda()
        with torch.no_grad(): vecs.append(model(t).cpu().numpy().ravel())
    df = pd.DataFrame(np.stack(vecs), index=[str(i) for i in ids],
                      columns=[f'fmcib_{j}' for j in range(len(vecs[0]))])
    df.index.name='patient'; df.to_parquet(out); print(out, df.shape)
embed_direct(f'{DIR}/fmcib_patches_discovery.npz', f'{DIR}/fmcib_embed_discovery.parquet')
embed_direct(f'{DIR}/fmcib_patches_rad_valid.npz', f'{DIR}/fmcib_embed_rad_valid.parquet')
```
(Cách trực tiếp dùng normalize xấp xỉ; nếu get_features chạy được thì ưu tiên nó vì đúng preprocessing fmcib.)